In [ ]:
%pip install python-docx pymupdf pillow charset-normalizer extract-msg anthropic

In [ ]:
%restart_python

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from docx import Document
from PIL import Image
import fitz  # PyMuPDF — layout-aware PDF text extraction + page rasterization
from charset_normalizer import from_bytes
import email as email_lib
from email import policy as email_policy
import extract_msg
import tempfile
from io import BytesIO
import os
import re
import json
import base64
import concurrent.futures
import anthropic

catalog = "cdac-project"
schema = "intelligent-main-folder"
metadata_table = f"`{catalog}`.`{schema}`.file_metadata"
classification_table = f"`{catalog}`.`{schema}`.document_classification"

# One schema per document type, matching the original Notebook 4/5/6
# layout — this notebook now writes structured data directly into
# these, since classification + field extraction happen here too.
DOCUMENT_TYPE_SCHEMAS = {
    "RESUME": "resume",
    "EMAIL": "email",
    "INVOICE": "invoice",
    "BANK_STATEMENT": "bank_statement",
    "PRESCRIPTION": "prescription",
}
for schema_name in DOCUMENT_TYPE_SCHEMAS.values():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema_name}`")

STRUCTURED_TABLES = {
    "RESUME": f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['RESUME']}`.resume_structured_data",
    "EMAIL": f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['EMAIL']}`.email_structured_data",
    "INVOICE": f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['INVOICE']}`.invoice_structured_data",
    "BANK_STATEMENT": f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['BANK_STATEMENT']}`.bank_statement_structured_data",
    "PRESCRIPTION": f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['PRESCRIPTION']}`.prescription_structured_data",
}

# Stored as a Databricks secret rather than pasted in the notebook —
# keeps the key out of notebook history / version control.
ANTHROPIC_API_KEY = dbutils.secrets.get(scope="cdac-project-secrets", key="anthropic-api-key")

In [ ]:
input_path = "/Volumes/cdac-project/intelligent-main-folder/raw"

In [ ]:
# ============================================================
# TEXT EXTRACTION — born-digital formats only (DOCX/TXT/email, and a
# PDF's own embedded text layer). These stay cheap, free, and
# reliable exactly as before — the only thing that changed is what
# happens to IMAGES and SCANNED PDF pages: those no longer go through
# Tesseract OCR at all. See the next cell — they're sent straight to
# Claude, which reads the picture directly instead of us guessing at
# a middle "extracted text" step first. This is what replaced OCR's
# unreliable output and the regex extractor's fragile exact-label
# matching, both real problems hit repeatedly on real documents.
# ============================================================

MIN_TEXT_LAYER_LENGTH = 20
# A genuinely scanned/image-only PDF page returns ~0 characters from
# the text layer. Below this length, the page is treated as needing
# the Claude-vision fallback instead of trusting a near-empty extract.


def extract_text_from_docx(content):
    document = Document(BytesIO(content))

    paragraphs = [p.text.strip() for p in document.paragraphs if p.text.strip()]

    tables_text = []
    for table in document.tables:
        for row in table.rows:
            seen_cells = set()
            row_cells = []
            for cell in row.cells:
                if id(cell._tc) in seen_cells:
                    continue
                seen_cells.add(id(cell._tc))
                cell_text = cell.text.strip()
                if cell_text:
                    row_cells.append(cell_text)
            if row_cells:
                tables_text.append(" | ".join(row_cells))

    return "\n".join(paragraphs + tables_text).strip()


def extract_text_from_txt(content):
    best_match = from_bytes(content).best()
    if best_match is None:
        raise Exception("Could not reliably determine the file's text encoding")
    return str(best_match).strip()


def strip_html_tags(html):
    text = re.sub(r"<[^>]+>", " ", html)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_eml(content):
    message = email_lib.message_from_bytes(content, policy=email_policy.default)

    sender = message.get("From")
    recipient = message.get("To")
    subject = message.get("Subject")
    date = message.get("Date")

    body = None
    if message.is_multipart():
        for part in message.walk():
            if part.get_content_type() == "text/plain":
                body = part.get_content()
                break
        else:
            for part in message.walk():
                if part.get_content_type() == "text/html":
                    body = strip_html_tags(part.get_content())
                    break
    else:
        body = message.get_content()

    return sender, recipient, subject, date, body


def parse_msg(content):
    with tempfile.NamedTemporaryFile(suffix=".msg") as tmp_file:
        tmp_file.write(content)
        tmp_file.flush()
        message = extract_msg.Message(tmp_file.name)
        sender = message.sender
        recipient = message.to
        subject = message.subject
        date = str(message.date) if message.date else None
        body = message.body

    return sender, recipient, subject, date, body


def extract_text_from_email(content, extension):
    if extension == "eml":
        sender, recipient, subject, date, body = parse_eml(content)
    else:
        sender, recipient, subject, date, body = parse_msg(content)

    header_and_body = [
        f"From: {sender or ''}",
        f"To: {recipient or ''}",
        f"Subject: {subject or ''}",
        f"Date: {date or ''}",
        "",
        body or "",
    ]
    return "\n".join(header_and_body).strip()


def extract_page_text_in_reading_order(page):
    """
    Reconstructs proper reading order for a page, including
    two-column layouts. See the original Notebook 2 history for why
    this exists (PyMuPDF's own block-clustering merges both columns'
    same-row text together otherwise) — logic unchanged.
    """

    words = page.get_text("words")
    if not words:
        return ""

    lines_by_key = {}
    for x0, y0, x1, y1, word, block_no, line_no, word_no in words:
        key = (block_no, line_no)
        line = lines_by_key.setdefault(key, {"x0": x0, "y0": y0, "x1": x1, "y1": y1, "words": []})
        line["x0"] = min(line["x0"], x0)
        line["y0"] = min(line["y0"], y0)
        line["x1"] = max(line["x1"], x1)
        line["y1"] = max(line["y1"], y1)
        line["words"].append((word_no, word))

    lines = []
    for line in lines_by_key.values():
        text = " ".join(w for _, w in sorted(line["words"]))
        lines.append((line["x0"], line["y0"], line["x1"], line["y1"], text))

    lines.sort(key=lambda l: l[1])

    midpoint = page.rect.width / 2
    left_column, right_column, full_width = [], [], []
    for line in lines:
        x0, x1 = line[0], line[2]
        if x1 <= midpoint:
            left_column.append(line)
        elif x0 >= midpoint:
            right_column.append(line)
        else:
            full_width.append(line)

    if not left_column or not right_column:
        return "\n".join(l[4] for l in lines)

    ordered_lines = []
    left_i = right_i = 0
    for full_width_line in full_width + [None]:
        boundary_y = full_width_line[1] if full_width_line else float("inf")
        while left_i < len(left_column) and left_column[left_i][1] < boundary_y:
            ordered_lines.append(left_column[left_i])
            left_i += 1
        while right_i < len(right_column) and right_column[right_i][1] < boundary_y:
            ordered_lines.append(right_column[right_i])
            right_i += 1
        if full_width_line:
            ordered_lines.append(full_width_line)

    return "\n".join(l[4] for l in ordered_lines)


def extract_pdf_text_layer(content):
    """
    Returns (joined_text, needs_vision_fallback). needs_vision_fallback
    is True if ANY page's embedded text layer is too short — a
    document with one real page and one scanned page shouldn't have
    the scanned page silently dropped just because the combined text
    cleared the length threshold.
    """

    doc = fitz.open(stream=content, filetype="pdf")
    try:
        pages_text = []
        needs_fallback = False
        for page in doc:
            page_text = extract_page_text_in_reading_order(page)
            pages_text.append(page_text)
            if len(page_text) < MIN_TEXT_LAYER_LENGTH:
                needs_fallback = True
        return "\n".join(pages_text).strip(), needs_fallback
    finally:
        doc.close()


def render_pdf_pages_as_images(content, dpi=200):
    """
    Renders every page of a PDF to a PNG image — used when the PDF's
    text layer is missing/too short (scanned document) instead of
    running Tesseract OCR. All pages are sent to Claude together in
    one call so it reads the whole document as a unit.
    """

    doc = fitz.open(stream=content, filetype="pdf")
    try:
        images = []
        for page in doc:
            pixmap = page.get_pixmap(dpi=dpi)
            png_bytes = pixmap.tobytes("png")
            images.append(("image/png", base64.b64encode(png_bytes).decode("utf-8")))
        return images
    finally:
        doc.close()

In [ ]:
# ============================================================
# CLASSIFICATION + FIELD EXTRACTION VIA CLAUDE
# ============================================================
# This is the real change from the old 3-notebook design (OCR ->
# keyword-classify -> regex-extract). One Claude call now reads the
# document (as text, as an image, or as a set of page images) and
# returns BOTH the document type AND every structured field in one
# shot. This fixed two separate, real problems seen on this project's
# actual documents:
#   - OCR quality on photographed/scanned documents was unreliable
#     (handwriting, skew, low light) — Claude reads the image
#     directly, no separate transcription step to go wrong.
#   - The regex extractor broke the moment a document used different
#     label wording than expected (e.g. "FOR" instead of "Patient
#     Name", "(Inscription)" instead of "Medications") — Claude
#     understands the document instead of pattern-matching text.
#
# Model: Haiku 4.5 — cheapest Claude model, verified fast and
# accurate enough for this task (~$0.004/document). Same model this
# project's backend instant-preview already uses.

CLAUDE_MODEL = "claude-haiku-4-5-20251001"
MAX_OUTPUT_TOKENS = 2000

# Claude's 10MB limit is checked against the BASE64-ENCODED string, not
# raw bytes (base64 inflates size by ~4/3). This threshold targets a RAW
# size whose base64 form comfortably clears 10,485,760 bytes. Same value
# as the website's instant-preview code (llm_extractor.py) — a real
# upload here (11.1MB raw) hit exactly this limit, got rejected by
# Claude, and the resulting error was silently swallowed by
# process_one_file's except-Exception and mislabeled "UNKNOWN" with no
# reason saved anywhere. This was the actual bug, not a document Claude
# couldn't recognize.
MAX_IMAGE_BYTES = 7_200_000

# Claude also rejects an image if EITHER dimension exceeds 8000px,
# independent of file size — confirmed by a real test run: a
# re-encoded JPEG stayed under 10MB but still got a 400 because its
# dimensions were too large. Byte-size alone isn't a sufficient check.
MAX_IMAGE_DIMENSION = 8000

EXTRACTION_PROMPT = """You are analyzing a document — either as plain text, as a single image, or as a set of page images from a multi-page scanned document (treat multiple images as one document, read them in order).

Step 1: Decide which ONE of these document types it is: RESUME, EMAIL, INVOICE, BANK_STATEMENT, PRESCRIPTION. If none clearly match, use UNKNOWN.

Step 2: Extract fields based on the type you chose. If given a multi-column layout, scan top to bottom in each column separately, do not skip any section.

- RESUME: name, email, phone, linkedin, github, skills (array of strings), education (array of strings), experience (array of strings), projects (array of strings), certifications (array of strings), summary
  - email/phone/linkedin/github are usually near the top or in a contact block — check carefully, they are easy to miss. Look specifically for: a line containing an "@" symbol (email), a line of digits often with dashes (phone), and any line near the contact info that looks like a website handle or URL (linkedin/github), even if it does not start with "http".
  - "skills" = ONLY individual skill/technology names or short skill phrases from a Skills section. NEVER put a job title, company name, or work description in "skills".
  - "experience" = one array entry PER JOB, and each entry must combine the job title, company name, and date range together in one string (e.g. "IT Management Supervisor at Langtown Community College (Sep 2018 - present)"), not just the bare job title alone.
  - "education" = one array entry PER DEGREE/CERTIFICATION found anywhere in an Education or Certifications section, including the institution and year if shown. Include ALL of them, do not stop after the first one or two.
  - "summary" = ONLY the introductory paragraph under a "Summary"/"Objective"/"About" heading. NEVER include contact info, education, or skills text inside "summary".
  - Do not include bullet symbols (•, -, ●, etc.) at the start of any list item — just the text itself.
- EMAIL: sender, recipient, subject, sent_date, body
- INVOICE: invoice_number, invoice_date, due_date, total_amount
- BANK_STATEMENT: account_number, statement_period, opening_balance, closing_balance, transactions (array of strings)
- PRESCRIPTION: patient_name, physician_name, prescription_date, diagnosis, medications (array of strings, include dosage/timing if visible)

Step 3: ALWAYS also include "additional_information" — an array of short strings capturing any OTHER meaningful information visible in the document that isn't already captured by the fields above (for example: a website, a blood group, a reference number, a policy number, a tax ID, a note or remark). Each entry should be a concise "Label: value" string. Do NOT repeat anything already captured in the fields above. Use an empty array if there is genuinely nothing else of note.

Output ONLY a single JSON object, no markdown code fences, no explanation, in exactly this shape:
{"document_type": "<TYPE>", "data": { ...only the fields listed above for that type, plus "additional_information"... }}

Use null for any text field that is not visible in the document, and an empty array for any list field with nothing found.
If document_type is UNKNOWN, set "data" to null.
"""

# A real bank statement upload hit "Expecting ',' delimiter" from
# json.loads — almost certainly an unescaped quote or a literal
# newline inside a transaction-description string (bank statement
# line items are exactly the kind of free text likely to contain
# both). Appended to the prompt as a retry, not baked into the
# original one, so the common case stays exactly as before.
JSON_RETRY_REMINDER = (
    "\n\nIMPORTANT: Your previous response could not be parsed as valid JSON. Output ONLY a "
    "single, strictly valid JSON object — escape every double quote and backslash inside "
    'string values (\\" and \\\\), and never include a literal newline inside a string value '
    "(use \\n instead). No text before or after the JSON."
)

FIELDS_BY_TYPE = {
    "RESUME": ["name", "email", "phone", "linkedin", "github", "skills", "education", "experience", "projects", "certifications", "summary"],
    "EMAIL": ["sender", "recipient", "subject", "sent_date", "body"],
    "INVOICE": ["invoice_number", "invoice_date", "due_date", "total_amount"],
    "BANK_STATEMENT": ["account_number", "statement_period", "opening_balance", "closing_balance", "transactions"],
    "PRESCRIPTION": ["patient_name", "physician_name", "prescription_date", "diagnosis", "medications"],
}

LIST_FIELDS = {"skills", "education", "experience", "projects", "certifications", "transactions", "medications", "additional_information"}


def parse_llm_json(raw_response):
    text = raw_response.strip()
    fence_match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    if fence_match:
        text = fence_match.group(1).strip()
    return json.loads(text)


def shape_data(document_type, raw_data):
    if document_type not in FIELDS_BY_TYPE or raw_data is None:
        return None

    # "additional_information" is appended here rather than duplicated
    # into every entry in FIELDS_BY_TYPE, since it's the one field
    # common to all 5 document types (see EXTRACTION_PROMPT Step 3).
    shaped = {"document_type": document_type}
    for field_name in [*FIELDS_BY_TYPE[document_type], "additional_information"]:
        value = raw_data.get(field_name)
        if field_name in LIST_FIELDS:
            shaped[field_name] = value if isinstance(value, list) else []
        else:
            shaped[field_name] = value if isinstance(value, str) and value.strip() else None
    return shaped


def detect_media_type(image_bytes):
    """
    Detects the image's real format from its bytes rather than the
    filename extension — a file uploaded as "photo.png" can genuinely
    contain WEBP data (common with browser screenshot tools), and
    Claude's API rejects a mismatch between declared and actual
    format. Hit this exact bug for real on this project.
    """

    with Image.open(BytesIO(image_bytes)) as image:
        pil_format = image.format

    media_type = {"JPEG": "image/jpeg", "PNG": "image/png", "GIF": "image/gif", "WEBP": "image/webp"}.get(pil_format or "")
    if media_type is None:
        raise ValueError(f"Unsupported image format: {pil_format}")
    return media_type


def shrink_image_if_needed(image_bytes, media_type):
    """
    Re-encodes as JPEG at progressively lower quality/resolution until
    it's under BOTH Claude's 10MB size limit AND its 8000px dimension
    limit — a file can be small in bytes but still too large in pixels
    (a highly compressed but very high-resolution photo), so both need
    checking, not just size. Only touches the image when it's actually
    over one of the two limits. Raises ValueError (caught by
    process_one_file, same as every other per-file error here) if it
    can't get small enough without becoming unreadable.
    """

    with Image.open(BytesIO(image_bytes)) as probe:
        original_dimension = max(probe.size)

    if len(image_bytes) <= MAX_IMAGE_BYTES and original_dimension <= MAX_IMAGE_DIMENSION:
        return image_bytes, media_type

    with Image.open(BytesIO(image_bytes)) as image:
        image = image.convert("RGB")

        max_dimension = min(max(image.size), MAX_IMAGE_DIMENSION)
        quality = 90

        while True:
            if max_dimension < max(image.size):
                scale = max_dimension / max(image.size)
                resized = image.resize((max(1, int(image.width * scale)), max(1, int(image.height * scale))))
            else:
                resized = image

            buffer = BytesIO()
            resized.save(buffer, format="JPEG", quality=quality)
            candidate = buffer.getvalue()

            if len(candidate) <= MAX_IMAGE_BYTES:
                return candidate, "image/jpeg"

            if quality > 50:
                quality -= 15
            else:
                max_dimension = int(max_dimension * 0.75)
                quality = 80

            if max_dimension < 400:
                raise ValueError("Image is too large to process, even after compressing it (limit: 10 MB).")


def build_claude_request(content, extension):
    """
    Returns (images, text) for one file — images is a list of
    (media_type, base64_data) tuples (possibly empty), text is a
    string or None. Exactly one of the two is meaningfully populated
    per file type.
    """

    extension = extension.lower().replace(".", "")

    if extension in ("jpg", "jpeg", "png"):
        media_type = detect_media_type(content)
        content, media_type = shrink_image_if_needed(content, media_type)
        return [(media_type, base64.b64encode(content).decode("utf-8"))], None

    elif extension == "pdf":
        text, needs_fallback = extract_pdf_text_layer(content)
        if needs_fallback:
            return render_pdf_pages_as_images(content), None
        return [], text

    elif extension == "docx":
        return [], extract_text_from_docx(content)

    elif extension == "txt":
        return [], extract_text_from_txt(content)

    elif extension in ("eml", "msg"):
        return [], extract_text_from_email(content, extension)

    else:
        raise ValueError(f"Unsupported file extension: {extension}")


def classify_and_extract(client, images, text):
    image_blocks = [
        {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": b64_data}}
        for media_type, b64_data in images
    ]

    prompt = EXTRACTION_PROMPT
    if text:
        prompt = prompt + "\n\nDocument text:\n" + text

    raw_text = ""
    parsed = None
    last_error = None

    # One retry with an explicit escaping reminder if the first
    # response isn't valid JSON — Claude's output isn't deterministic,
    # so this usually gets a clean parse the second time rather than
    # failing the whole file over a transient formatting slip.
    for attempt, attempt_prompt in enumerate((prompt, prompt + JSON_RETRY_REMINDER)):
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=MAX_OUTPUT_TOKENS,
            messages=[{"role": "user", "content": [*image_blocks, {"type": "text", "text": attempt_prompt}]}],
        )
        raw_text = "".join(block.text for block in response.content if block.type == "text")

        try:
            parsed = parse_llm_json(raw_text)
            break
        except json.JSONDecodeError as e:
            last_error = e
            print(f"Malformed JSON on attempt {attempt + 1}: {e} | raw response (truncated): {raw_text[:2000]}")

    if parsed is None:
        raise last_error

    document_type = parsed.get("document_type", "UNKNOWN")
    if document_type not in FIELDS_BY_TYPE:
        document_type = "UNKNOWN"

    data = shape_data(document_type, parsed.get("data"))
    return document_type, data


def process_one_file(file_id, file_name, content, extension, api_key):
    """
    Runs end-to-end for ONE file: build the Claude request (text or
    image(s)), call Claude, return a plain dict. Never raises — a bad
    file shouldn't take down the whole parallel batch, same principle
    as every other per-record error handling in this pipeline.
    """

    try:
        images, text = build_claude_request(content, extension)
        client = anthropic.Anthropic(api_key=api_key)
        document_type, data = classify_and_extract(client, images, text)
        return {
            "file_id": file_id, "file_name": file_name,
            "document_type": document_type, "data": data,
            "status": "SUCCESS", "error_message": None,
        }
    except Exception as e:
        return {
            "file_id": file_id, "file_name": file_name,
            "document_type": "UNKNOWN", "data": None,
            "status": "FAILED", "error_message": str(e),
        }

In [ ]:
# Binary content is read here, sequentially, BEFORE any parallel
# work starts — Spark reads aren't meant to be called concurrently
# from worker threads, so this hands the thread pool below plain
# Python bytes to work with, not Spark calls.

metadata_df = spark.table(metadata_table).filter(F.col("status") == "NEW")
new_files_rows = metadata_df.collect()

file_content_by_id = {}
for row in new_files_rows:
    content = spark.read.format("binaryFile").load(row["file_path"]).collect()[0]["content"]
    file_content_by_id[row["file_id"]] = content

print(f"{len(new_files_rows)} new file(s) to process")

In [ ]:
# Claude calls are I/O-bound (waiting on the network), not CPU-bound,
# so running them concurrently instead of one-by-one is a real speed
# win: ~10 files sequentially is roughly 10x a single call's latency,
# concurrently it's roughly ONE call's latency for the whole batch.
# Capped at 10 — comfortably under Haiku 4.5's rate limit (1000
# requests/minute) and matches this project's realistic batch size.

MAX_WORKERS = 10

results = []
if new_files_rows:
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(
                process_one_file,
                row["file_id"], row["file_name"],
                file_content_by_id[row["file_id"]],
                row["file_extension"],
                ANTHROPIC_API_KEY,
            ): row["file_id"]
            for row in new_files_rows
        }
        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())

for r in results:
    if r["status"] == "FAILED":
        print(f"{r['file_name']}: {r['document_type']} ({r['status']}) -- {r['error_message']}")
    else:
        print(f"{r['file_name']}: {r['document_type']} ({r['status']})")

In [ ]:
# Written in the exact same shape the old keyword-classifier notebook
# used, so nothing downstream (this project's backend query, or the
# rest of this pipeline) needs to change — only HOW document_type
# gets decided changed, not the table shape.
#
# raw_text is left null for image/scanned-PDF sources — Claude reads
# those directly with no separate transcription step, so there's no
# plain-text version to store. Text-based sources (DOCX/TXT/PDF text
# layer/email) still have real text available and it's optional to
# thread through here; kept null for all rows for simplicity, matching
# what the backend's own instant-preview path already does.

classification_schema = """
file_id STRING,
file_name STRING,
raw_text STRING,
document_type STRING,
classification_confidence DOUBLE,
status STRING
"""

classification_rows = []
for r in results:
    document_type = r["document_type"]
    status = "CLASSIFIED" if document_type != "UNKNOWN" else "UNCLASSIFIED"
    confidence = 1.0 if document_type != "UNKNOWN" else 0.0
    classification_rows.append((r["file_id"], r["file_name"], None, document_type, confidence, status))

if classification_rows:
    classification_df = (
        spark.createDataFrame(classification_rows, schema=classification_schema)
        .withColumn("classification_timestamp", F.current_timestamp())
    )
    classification_df.write.format("delta").mode("append").saveAsTable(classification_table)

display(spark.table(classification_table))

In [ ]:
STRUCTURED_SCHEMAS = {
    "RESUME": "file_id STRING, document_type STRING, name STRING, email STRING, phone STRING, linkedin STRING, github STRING, skills ARRAY<STRING>, education ARRAY<STRING>, experience ARRAY<STRING>, projects ARRAY<STRING>, certifications ARRAY<STRING>, summary STRING, additional_information ARRAY<STRING>, processing_status STRING, error_message STRING",
    "EMAIL": "file_id STRING, document_type STRING, sender STRING, recipient STRING, subject STRING, sent_date STRING, body STRING, additional_information ARRAY<STRING>, processing_status STRING, error_message STRING",
    "INVOICE": "file_id STRING, document_type STRING, invoice_number STRING, invoice_date STRING, due_date STRING, total_amount STRING, additional_information ARRAY<STRING>, processing_status STRING, error_message STRING",
    "BANK_STATEMENT": "file_id STRING, document_type STRING, account_number STRING, statement_period STRING, opening_balance STRING, closing_balance STRING, transactions ARRAY<STRING>, additional_information ARRAY<STRING>, processing_status STRING, error_message STRING",
    "PRESCRIPTION": "file_id STRING, document_type STRING, patient_name STRING, physician_name STRING, prescription_date STRING, diagnosis STRING, medications ARRAY<STRING>, additional_information ARRAY<STRING>, processing_status STRING, error_message STRING",
}

structured_rows_by_type = {t: [] for t in FIELDS_BY_TYPE}
for r in results:
    if r["status"] == "SUCCESS" and r["data"] is not None:
        structured_rows_by_type[r["document_type"]].append(r)

for document_type, rows in structured_rows_by_type.items():
    if not rows:
        continue

    # "additional_information" isn't in FIELDS_BY_TYPE (it's common to
    # every type, appended by shape_data instead — see the previous
    # cell), so it's added here explicitly rather than duplicated into
    # every type's field list.
    field_names = [*FIELDS_BY_TYPE[document_type], "additional_information"]
    row_tuples = [
        (r["file_id"], document_type, *[r["data"].get(f) for f in field_names], "SUCCESS", None)
        for r in rows
    ]

    df = (
        spark.createDataFrame(row_tuples, schema=STRUCTURED_SCHEMAS[document_type])
        .withColumn("processing_timestamp", F.current_timestamp())
    )
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(STRUCTURED_TABLES[document_type])
    print(f"Wrote {len(rows)} {document_type} record(s)")

In [ ]:
# Marks each file PROCESSED/FAILED so the next run's `status == "NEW"`
# filter doesn't pick it up again — same purpose as the original
# Notebook 2's status-update step, just driven by this notebook's own
# results now.

# error_message lets a failure like "image too large" or a transient
# API error show up in the table itself instead of only existing (or
# not, if caught and discarded like before) in a job run's console
# logs. Column is added on first run only — ADD COLUMNS errors out if
# the column already exists, so this checks first.
if "error_message" not in spark.table(metadata_table).columns:
    spark.sql(f"ALTER TABLE {metadata_table} ADD COLUMNS (error_message STRING)")

status_updates = [
    (r["file_id"], "PROCESSED" if r["status"] == "SUCCESS" else "FAILED", r["error_message"])
    for r in results
]

if status_updates:
    # Explicit schema, not just column names — when every row's
    # error_message happens to be None (a batch with no failures),
    # Spark can't infer that column's type from all-null values and
    # throws CANNOT_DETERMINE_TYPE. Hit this for real on a 1-file batch.
    status_df = spark.createDataFrame(status_updates, "file_id STRING, status STRING, error_message STRING")
    metadata_delta_table = DeltaTable.forName(spark, metadata_table)
    (
        metadata_delta_table.alias("meta")
        .merge(status_df.alias("upd"), "meta.file_id = upd.file_id")
        .whenMatchedUpdate(set={"status": "upd.status", "error_message": "upd.error_message"})
        .execute()
    )

display(spark.table(metadata_table))

In [ ]:
for document_type, table_name in STRUCTURED_TABLES.items():
    if spark.catalog.tableExists(table_name):
        print(f"--- {document_type} ---")
        display(spark.table(table_name))